# Spark + Iceberg Local Analysis

Simple examples for working with Spark and Iceberg tables.

In [1]:
from pyspark.sql import SparkSession

# Connect to Spark
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()
print("Connected to Spark!")

Connected to Spark!


In [2]:
# Show available databases
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|     demo|
+---------+



In [3]:
# Create a database
spark.sql("CREATE DATABASE IF NOT EXISTS demo").show()

++
||
++
++



In [4]:
# Create an Iceberg table
spark.sql("""
    CREATE TABLE IF NOT EXISTS demo.sales (
        id INT,
        product STRING,
        amount DOUBLE,
        sale_date DATE
    ) USING iceberg
""").show()

++
||
++
++



In [5]:
# Insert some data (use DATE() function to properly cast date strings)
spark.sql("""
    INSERT INTO demo.sales VALUES 
    (1, 'Widget A', 99.99, DATE('2026-02-22')),
    (2, 'Widget B', 149.99, DATE('2026-02-22')),
    (3, 'Widget C', 199.99, DATE('2026-02-22'))
""").show()

++
||
++
++



In [6]:
# Query the table
spark.sql("SELECT * FROM demo.sales").show()

+---+--------+------+----------+
| id| product|amount| sale_date|
+---+--------+------+----------+
|  1|Widget A| 99.99|2026-02-22|
|  1|Widget A| 99.99|2026-02-22|
|  2|Widget B|149.99|2026-02-22|
|  2|Widget B|149.99|2026-02-22|
|  3|Widget C|199.99|2026-02-22|
|  3|Widget C|199.99|2026-02-22|
+---+--------+------+----------+



In [7]:
# Use DataFrame API
df = spark.table("demo.sales")
df.filter(df.amount > 100).show()

+---+--------+------+----------+
| id| product|amount| sale_date|
+---+--------+------+----------+
|  2|Widget B|149.99|2026-02-22|
|  3|Widget C|199.99|2026-02-22|
|  2|Widget B|149.99|2026-02-22|
|  3|Widget C|199.99|2026-02-22|
+---+--------+------+----------+



In [8]:
# Iceberg features - view table history
spark.sql("SELECT * FROM demo.sales.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-02-22 13:33:59.055|6457575407755750857|NULL               |true               |
|2026-02-22 13:35:46.927|9081339521484012374|6457575407755750857|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [9]:
# View snapshots
spark.sql("SELECT * FROM demo.sales.snapshots").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                     |summary                                                                                                                                                         

In [10]:
# Time travel - query previous snapshot
# Get the first available snapshot ID
snapshots = spark.sql("SELECT snapshot_id FROM demo.sales.snapshots ORDER BY committed_at").collect()
if len(snapshots) > 0:
    first_snapshot = snapshots[0].snapshot_id
    print(f"Querying snapshot ID: {first_snapshot}")
    spark.sql(f"SELECT * FROM demo.sales VERSION AS OF {first_snapshot}").show()
else:
    print("No snapshots available yet")

# Alternative: Time travel using timestamp (query data from 5 minutes ago)
from datetime import datetime, timedelta
past_time = (datetime.now() - timedelta(minutes=5)).strftime('%Y-%m-%d %H:%M:%S')
try:
    print(f"\nQuerying data as of: {past_time}")
    spark.sql(f"SELECT * FROM demo.sales TIMESTAMP AS OF '{past_time}'").show()
except Exception as e:
    print(f"Note: Timestamp is before table creation: {e}")

Querying snapshot ID: 6457575407755750857
+---+--------+------+----------+
| id| product|amount| sale_date|
+---+--------+------+----------+
|  1|Widget A| 99.99|2026-02-22|
|  2|Widget B|149.99|2026-02-22|
|  3|Widget C|199.99|2026-02-22|
+---+--------+------+----------+


Querying data as of: 2026-02-22 14:30:48
+---+--------+------+----------+
| id| product|amount| sale_date|
+---+--------+------+----------+
|  1|Widget A| 99.99|2026-02-22|
|  2|Widget B|149.99|2026-02-22|
|  3|Widget C|199.99|2026-02-22|
|  1|Widget A| 99.99|2026-02-22|
|  2|Widget B|149.99|2026-02-22|
|  3|Widget C|199.99|2026-02-22|
+---+--------+------+----------+



In [11]:
# Show all tables
spark.sql("SHOW TABLES IN demo").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     demo|    sales|      false|
+---------+---------+-----------+

